# 🛰️ Hands-On: Publish/Subscribe Berbasis MQTT

**Tujuan Pembelajaran**
Setelah menyelesaikan latihan ini, mahasiswa mampu:
- Menjelaskan cara kerja pola komunikasi *Publish/Subscribe*
- Menghubungkan klien Python ke broker MQTT
- Mempublikasikan pesan ke topik tertentu
- Berlangganan (*subscribe*) topik dan menerima pesan secara real-time
- Menggunakan wildcard dan memahami level QoS

---

## Prasyarat

Pastikan **broker MQTT sudah berjalan** di mesin lokal:

```bash
cd /home/momotaro/pub-sub
sudo docker compose up -d
```

Broker dapat diakses di `localhost:1883`.


---
## Bagian 1 — Instalasi Library

Library yang digunakan: **paho-mqtt** (klien MQTT resmi untuk Python).

In [ ]:
!pip install paho-mqtt --quiet
print("paho-mqtt siap digunakan ✓")

---
## Bagian 2 — Konsep Pub/Sub & MQTT

### Cara Kerja

```
Publisher  ──► [Topik: sensor/suhu] ──►  Broker  ──►  Subscriber-A
                                                  ──►  Subscriber-B
```

| Istilah | Penjelasan |
|---------|-----------|
| **Broker** | Server perantara yang menyimpan dan meneruskan pesan |
| **Publisher** | Klien yang *mengirim* pesan ke sebuah topik |
| **Subscriber** | Klien yang *mendaftar* ke topik dan menerima pesan |
| **Topik** | Label string hierarkis, misal `coba/pesan` atau `sensor/suhu` |
| **QoS** | Jaminan pengiriman: 0 = at most once, 1 = at least once, 2 = exactly once |

### Wildcard Topik

| Wildcard | Arti | Contoh |
|----------|------|--------|
| `+` | Satu level apa saja | `sensor/+` cocok dengan `sensor/suhu`, `sensor/cahaya` |
| `#` | Semua level di bawahnya | `sensor/#` cocok dengan `sensor/suhu/lantai1` dst |

---
## Bagian 3 — Konfigurasi Koneksi

Sebelum mulai, jalankan sel berikut untuk menyiapkan variabel koneksi.

In [ ]:
import paho.mqtt.client as mqtt
import json, time, threading
from datetime import datetime

# ── Ubah nilai ini sesuai lingkungan Anda ──────────────────────────────
BROKER_HOST = "localhost"   # ganti dengan IP server jika broker di mesin lain
BROKER_PORT = 1883
# ───────────────────────────────────────────────────────────────────────

print(f"Akan terhubung ke broker  {BROKER_HOST}:{BROKER_PORT}")

---
## Latihan 1 — Subscribe: Mendengarkan Topik yang Sudah Berjalan

Broker sudah menerima data dari kontainer `publisher` setiap 2 detik.  
Tugas Anda: **buat subscriber** yang menerima pesan tersebut.

> **Cara baca output:** Sel di bawah akan *memblokir* selama `durasi_detik` detik  
> lalu berhenti otomatis. Anda akan melihat setiap pesan yang masuk.

In [ ]:
# ── Latihan 1: Subscribe ke sensor/suhu ───────────────────────────────
TOPIK_L1   = "sensor/suhu"   # ← coba ubah ke "sensor/#" dan amati bedanya
durasi_detik = 10             # ← berapa detik mendengarkan?

pesan_diterima = []

def saat_terhubung(client, userdata, flags, rc, props):
    print(f"✓ Terhubung ke broker (rc={rc})")
    client.subscribe(TOPIK_L1, qos=1)
    print(f"  Berlangganan topik: '{TOPIK_L1}'")

def saat_pesan_masuk(client, userdata, msg):
    data = json.loads(msg.payload.decode())
    pesan_diterima.append(data)
    print(f"  📨 [{msg.topic}]  suhu={data['suhu_c']}°C  "
          f"kelembaban={data['kelembaban']}%  waktu={data['timestamp']}")

sub1 = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="latihan1-sub")
sub1.on_connect = saat_terhubung
sub1.on_message = saat_pesan_masuk

sub1.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
sub1.loop_start()

print(f"Mendengarkan selama {durasi_detik} detik …")
time.sleep(durasi_detik)
sub1.loop_stop()
sub1.disconnect()
print(f"\nSelesai. Total pesan diterima: {len(pesan_diterima)}")

---
## Latihan 2 — Publish: Mengirim Pesan Sendiri

Sekarang Anda menjadi **publisher**. Kirimkan pesan ke topik buatan Anda sendiri.

In [ ]:
# ── Latihan 2: Publish pesan ke topik Anda sendiri ────────────────────
NAMA_ANDA  = "mahasiswa"      # ← ganti dengan nama/NIM Anda
TOPIK_L2   = f"kelas/{NAMA_ANDA}"

pesan_kirim = [
    {"isi": "Halo dari Jupyter!", "urutan": 1},
    {"isi": "Belajar MQTT itu mudah", "urutan": 2},
    {"isi": "Pesan terakhir", "urutan": 3},
]

def saat_publish(client, userdata, mid, rc, props):
    print(f"  ✓ Pesan terkirim (mid={mid})")

pub2 = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=f"latihan2-{NAMA_ANDA}")
pub2.on_publish = saat_publish
pub2.connect(BROKER_HOST, BROKER_PORT)
pub2.loop_start()

print(f"Mengirim {len(pesan_kirim)} pesan ke topik '{TOPIK_L2}' …\n")
for p in pesan_kirim:
    result = pub2.publish(TOPIK_L2, json.dumps(p), qos=1)
    print(f"  → {p}")
    time.sleep(0.5)

pub2.loop_stop()
pub2.disconnect()
print("\nSemua pesan terkirim!")
print(f"\n💡 Coba subscribe ke '{TOPIK_L2}' dari terminal lain:\n"
      f"   mosquitto_sub -h {BROKER_HOST} -t '{TOPIK_L2}' -v")

---
## Latihan 3 — Publish ke Banyak Topik + Wildcard Subscriber

Anda akan:
1. **Mempublikasikan** pesan ke tiga topik berbeda secara bersamaan
2. **Berlangganan** dengan wildcard `+` dan `#`, lalu amati perbedaannya

| Topik dikirim | `sensor/+` | `sensor/#` |
|---|:---:|:---:|
| `sensor/suhu` | ✔ | ✔ |
| `sensor/cahaya` | ✔ | ✔ |
| `sensor/suhu/lantai1` | ✘ | ✔ |

In [ ]:
# ── Latihan 3: Wildcard ───────────────────────────────────────────────
WILDCARD    = "sensor/#"      # ← coba ganti ke "sensor/+" dan amati bedanya
TOPIK_L3    = ["sensor/suhu", "sensor/cahaya", "sensor/suhu/lantai1"]
durasi_l3   = 8

diterima_l3 = []

def on_connect_l3(client, userdata, flags, rc, props):
    print(f"✓ Subscriber terhubung, berlangganan '{WILDCARD}'")
    client.subscribe(WILDCARD, qos=0)

def on_message_l3(client, userdata, msg):
    payload = json.loads(msg.payload.decode())
    diterima_l3.append(msg.topic)
    print(f"  📨 [{msg.topic}]  {payload}")

# ── Subscriber dengan wildcard
sub3 = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="latihan3-sub")
sub3.on_connect = on_connect_l3
sub3.on_message = on_message_l3
sub3.connect(BROKER_HOST, BROKER_PORT)
sub3.loop_start()

time.sleep(1)  # tunggu sambungan subscriber siap

# ── Publisher: kirim ke 3 topik
pub3 = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="latihan3-pub")
pub3.connect(BROKER_HOST, BROKER_PORT)
pub3.loop_start()

print(f"\nMengirim pesan ke {len(TOPIK_L3)} topik …")
for t in TOPIK_L3:
    data = {"topik_asal": t, "waktu": datetime.utcnow().isoformat()}
    pub3.publish(t, json.dumps(data), qos=0)
    print(f"  → {t}")
    time.sleep(0.5)

time.sleep(durasi_l3)
pub3.loop_stop(); pub3.disconnect()
sub3.loop_stop(); sub3.disconnect()

print(f"\n📊 Ringkasan (wildcard='{WILDCARD}'):")
for t in TOPIK_L3:
    status = "✔ diterima" if t in diterima_l3 else "✘ tidak diterima"
    print(f"   {t:35s}  {status}")

print("\n💡 Ganti WILDCARD ke 'sensor/+' dan jalankan ulang untuk melihat perbedaan!")

---
## Latihan 4 — Memahami Level QoS

MQTT memiliki tiga level **Quality of Service (QoS)**:

| QoS | Nama | Jaminan | Overhead |
|-----|------|---------|----------|
| **0** | At most once | Paling banyak 1x (bisa hilang) | Rendah |
| **1** | At least once | Minimal 1x (bisa duplikat) | Sedang |
| **2** | Exactly once | Tepat 1x | Tinggi |

Sel berikut mepublikasikan 5 pesan dengan masing-masing level QoS dan menghitung berapa yang berhasil diterima.

In [ ]:
# ── Latihan 4: Perbandingan QoS ──────────────────────────────────────
JUMLAH_PESAN = 5
hasil_qos    = {}

for qos_level in [0, 1, 2]:
    topik_qos  = f"qos/level{qos_level}"
    counter    = {"diterima": 0}
    event_done = threading.Event()

    def _on_connect(client, userdata, flags, rc, props, _t=topik_qos, _q=qos_level):
        client.subscribe(_t, qos=_q)

    def _on_message(client, userdata, msg, _c=counter, _e=event_done):
        _c["diterima"] += 1
        if _c["diterima"] >= JUMLAH_PESAN:
            _e.set()

    sub_qos = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2,
                          client_id=f"sub-qos{qos_level}")
    sub_qos.on_connect = _on_connect
    sub_qos.on_message = _on_message
    sub_qos.connect(BROKER_HOST, BROKER_PORT)
    sub_qos.loop_start()
    time.sleep(0.5)

    pub_qos = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2,
                          client_id=f"pub-qos{qos_level}")
    pub_qos.connect(BROKER_HOST, BROKER_PORT)
    pub_qos.loop_start()

    for i in range(JUMLAH_PESAN):
        pub_qos.publish(topik_qos,
                        json.dumps({"nomor": i+1, "qos": qos_level}),
                        qos=qos_level)
        time.sleep(0.1)

    event_done.wait(timeout=5)
    pub_qos.loop_stop(); pub_qos.disconnect()
    sub_qos.loop_stop(); sub_qos.disconnect()

    hasil_qos[qos_level] = counter["diterima"]
    print(f"  QoS {qos_level}: {counter['diterima']}/{JUMLAH_PESAN} pesan diterima")

print("\n📊 Ringkasan QoS:")
for lvl, jumlah in hasil_qos.items():
    label = ["at most once", "at least once", "exactly once"][lvl]
    print(f"   QoS {lvl} ({label:14s}):  {jumlah}/{JUMLAH_PESAN} pesan")

---
## Latihan 5 — Simulasi Chat Sederhana via MQTT

Dua "pengguna" (A dan B) saling berkirim pesan melalui topik `chat/ruangan`.  
Ini mensimulasikan *group chat* sederhana berbasis pub/sub.

In [ ]:
# ── Latihan 5: Simulasi Chat ──────────────────────────────────────────
TOPIK_CHAT = "chat/ruangan"

# Riwayat percakapan yang akan dikirim
percakapan = [
    ("A", "Halo semua!"),
    ("B", "Hai A, apa kabar?"),
    ("A", "Baik! Sedang belajar MQTT"),
    ("B", "Seru! Pub/Sub itu powerful ya"),
    ("A", "Betul, satu publisher bisa ke banyak subscriber"),
]

log_chat = []

def on_chat_connect(client, userdata, flags, rc, props):
    client.subscribe(TOPIK_CHAT, qos=1)

def on_chat_message(client, userdata, msg):
    data = json.loads(msg.payload.decode())
    log_chat.append(data)
    print(f"  💬 [{data['waktu']}] {data['pengirim']:>2s}: {data['pesan']}")

# ── Buat satu subscriber yang "mendengarkan ruangan"
monitor = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="chat-monitor")
monitor.on_connect = on_chat_connect
monitor.on_message = on_chat_message
monitor.connect(BROKER_HOST, BROKER_PORT)
monitor.loop_start()
time.sleep(0.5)

# ── Simulasikan dua pengguna mengirim pesan bergantian
pub_chat = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="chat-pub")
pub_chat.connect(BROKER_HOST, BROKER_PORT)
pub_chat.loop_start()

print("=== Ruang Chat: chat/ruangan ===\n")
for pengirim, pesan in percakapan:
    payload = {
        "pengirim": pengirim,
        "pesan":    pesan,
        "waktu":    datetime.utcnow().strftime("%H:%M:%S"),
    }
    pub_chat.publish(TOPIK_CHAT, json.dumps(payload), qos=1)
    time.sleep(0.8)

time.sleep(1)
pub_chat.loop_stop(); pub_chat.disconnect()
monitor.loop_stop(); monitor.disconnect()

print(f"\nTotal pesan di log: {len(log_chat)}")
print("\n💡 Coba subscribe ke topik ini dari browser menggunakan WebSocket (port 9001)!")

---
## 🏆 Tantangan Mandiri

Pilih salah satu atau lebih tantangan di bawah ini:

### Tantangan A — Topik Bertingkat
Publish data ke topik bertingkat sesuai NIM Anda:
```
gedung/lantai1/<NIM>/suhu
gedung/lantai1/<NIM>/cahaya
gedung/lantai2/<NIM>/suhu
```
Kemudian buat subscriber yang hanya menerima pesan dari lantai 1 (`gedung/lantai1/#`).

---

### Tantangan B — Retained Message
Tambahkan `retain=True` saat publish. Lalu buka subscriber *setelah* publisher selesai — apakah pesan tetap diterima? Mengapa?

```python
client.publish("topik/saya", "pesan ini disimpan", retain=True)
```

---

### Tantangan C — Last Will & Testament (LWT)
Konfigurasikan *will message* pada client sehingga broker otomatis mengirim pesan ke topik tertentu ketika client tiba-tiba terputus.

```python
client.will_set("status/<nama>", payload="offline", qos=1, retain=True)
```

---

### Tantangan D — Dashboard Sederhana
Buat subscriber yang mengumpulkan 20 data suhu dari topik `sensor/suhu`, lalu hitung dan tampilkan:
- Rata-rata suhu
- Suhu tertinggi & terendah
- Histogram distribusi (gunakan `collections.Counter` atau `matplotlib`)

---

> **💡 Petunjuk:** Semua tantangan ini bisa diselesaikan hanya dengan memodifikasi sel-sel yang sudah ada di notebook ini.

In [ ]:
# ── Starter code untuk Tantangan D — silakan lengkapi! ───────────────
TARGET_PESAN = 20
data_suhu    = []
selesai      = threading.Event()

def on_connect_td(client, userdata, flags, rc, props):
    client.subscribe("sensor/suhu", qos=1)
    print(f"Mengumpulkan {TARGET_PESAN} data suhu …")

def on_message_td(client, userdata, msg):
    payload = json.loads(msg.payload.decode())
    data_suhu.append(payload["suhu_c"])
    print(f"  [{len(data_suhu):>2}/{TARGET_PESAN}] suhu={payload['suhu_c']}°C")
    if len(data_suhu) >= TARGET_PESAN:
        selesai.set()

sub_td = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="tantangan-d")
sub_td.on_connect = on_connect_td
sub_td.on_message = on_message_td
sub_td.connect(BROKER_HOST, BROKER_PORT)
sub_td.loop_start()

selesai.wait(timeout=60)
sub_td.loop_stop()
sub_td.disconnect()

if data_suhu:
    rata  = sum(data_suhu) / len(data_suhu)
    print(f"\n📊 Statistik dari {len(data_suhu)} data:")
    print(f"   Rata-rata : {rata:.2f}°C")
    print(f"   Tertinggi : {max(data_suhu):.2f}°C")
    print(f"   Terendah  : {min(data_suhu):.2f}°C")

    # TODO Tantangan D: tambahkan histogram di sini menggunakan matplotlib
else:
    print("Tidak ada data yang terkumpul — pastikan publisher berjalan!")